# Thesis batch-experiment analysis

Consumes the four `*_batch_*.csv` files produced by `ExperimentRunner` and renders thesis figures.

Auto-picks the newest batch timestamp on Desktop. Override `BATCH_DIR` / `BATCH_TIMESTAMP` below.

In [ ]:
import os, glob, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='paper')

BATCH_DIR = Path(os.path.expanduser('~')) / 'Desktop'
BATCH_TIMESTAMP = None  # e.g. '2026-04-23_14-22-05'; None = newest on disk
FIG_DIR = Path('figures'); FIG_DIR.mkdir(exist_ok=True)

CONDITION_ORDER = ['PureBOIDS', 'BOIDSWithGOAPLeader', 'GOAPWithBOIDSMovement']
PALETTE = dict(zip(CONDITION_ORDER, sns.color_palette('Set2', n_colors=3)))

In [ ]:
def _resolve_timestamp():
    if BATCH_TIMESTAMP:
        return BATCH_TIMESTAMP
    files = glob.glob(str(BATCH_DIR / 'TrialSummary_batch_*.csv'))
    if not files:
        raise FileNotFoundError(f'No TrialSummary_batch_*.csv in {BATCH_DIR}')
    newest = max(files, key=os.path.getmtime)
    m = re.search(r'TrialSummary_batch_(.+)\.csv$', newest)
    return m.group(1)

TS = _resolve_timestamp()
print(f'Loading batch timestamp: {TS}')

def _load(prefix):
    return pd.read_csv(BATCH_DIR / f'{prefix}_batch_{TS}.csv')

def _load_eventlog():
    path = BATCH_DIR / f'EventLog_batch_{TS}.csv'
    cols = ['Time','Condition','RunId','Seed','EventType','AgentName','FlockId','Details']
    with open(path, 'r', encoding='utf-8') as f:
        next(f)
        rows = [ln.rstrip().split(',', 7) for ln in f if ln.strip()]
    df = pd.DataFrame(rows, columns=cols)
    df['Time'] = pd.to_numeric(df['Time'], errors='coerce')
    for c in ('RunId','Seed','FlockId'):
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
    return df

trials = _load('TrialSummary')
perf   = _load('Performance')
goals  = _load('GoalDistribution')
events = _load_eventlog()

for name, df in [('trials', trials), ('perf', perf), ('goals', goals), ('events', events)]:
    print(f'{name}: {len(df):>7} rows, cols={list(df.columns)}')

trials['Condition']   = pd.Categorical(trials['Condition'],   categories=CONDITION_ORDER, ordered=True)
perf['Condition']     = pd.Categorical(perf['Condition'],     categories=CONDITION_ORDER, ordered=True)
goals['Condition']    = pd.Categorical(goals['Condition'],    categories=CONDITION_ORDER, ordered=True)
trials.head()


## 1 â€” Mean-metric bar chart per condition, faceted by AgentCount

In [ ]:
METRICS = ['AvgFPS', 'AvgCpuMs', 'AvgCpuMsPerAgent',
           'ReactionMeleeMs', 'ReactionRangedMs', 'GoalEntropy', 'DurationSec']

t = trials.copy()
for col in ('ReactionMeleeMs', 'ReactionRangedMs'):
    t.loc[t[col] < 0, col] = np.nan  # -1 sentinel = never reacted

sizes = sorted(t['AgentCount'].unique())
fig, axes = plt.subplots(len(METRICS), len(sizes),
                          figsize=(3 * len(sizes), 2.2 * len(METRICS)),
                          sharex='col', squeeze=False)
for i, m in enumerate(METRICS):
    for j, sz in enumerate(sizes):
        sub = t[t['AgentCount'] == sz]
        ax = axes[i, j]
        sns.barplot(data=sub, x='Condition', y=m, hue='Condition',
                    palette=PALETTE, errorbar='sd', ax=ax, legend=False)
        ax.set_title(f'N={sz}' if i == 0 else '')
        if j == 0: ax.set_ylabel(m)
        else:      ax.set_ylabel('')
        ax.tick_params(axis='x', rotation=30)
fig.suptitle('Mean metrics per condition (error bars = 1Ïƒ)')
fig.tight_layout()
fig.savefig(FIG_DIR / f'1_mean_metrics_{TS}.png', dpi=150, bbox_inches='tight')
plt.show()

## 2 â€” Goal-entropy box plot

In [ ]:
fig, ax = plt.subplots(figsize=(max(6, 2 * len(sizes)), 4))
sns.boxplot(data=trials, x='AgentCount', y='GoalEntropy',
            hue='Condition', palette=PALETTE, ax=ax)
ax.set_title('Goal entropy (Shannon, 9-bucket) per condition Ã— size')
ax.set_ylabel('Entropy (bits)')
fig.tight_layout()
fig.savefig(FIG_DIR / f'2_entropy_boxplot_{TS}.png', dpi=150, bbox_inches='tight')
plt.show()

## 3 â€” Reaction-time histogram (melee + ranged)

In [ ]:
long = t.melt(id_vars=['Condition', 'AgentCount'],
              value_vars=['ReactionMeleeMs', 'ReactionRangedMs'],
              var_name='FlockType', value_name='ReactionMs').dropna(subset=['ReactionMs'])
long['FlockType'] = long['FlockType'].str.replace('Reaction', '').str.replace('Ms', '')

g = sns.displot(long, x='ReactionMs', hue='Condition', col='FlockType',
                palette=PALETTE, kind='hist', bins=30, element='step',
                common_norm=False, stat='count', height=3.5, aspect=1.2)
g.fig.suptitle('Reaction-time distribution (stimulus â†’ first response)', y=1.02)
g.savefig(FIG_DIR / f'3_reaction_hist_{TS}.png', dpi=150, bbox_inches='tight')
plt.show()

censored = trials[['Condition', 'AgentCount', 'ReactionMeleeMs', 'ReactionRangedMs']].copy()
censored['MeleeCensored']  = censored['ReactionMeleeMs']  < 0
censored['RangedCensored'] = censored['ReactionRangedMs'] < 0
print('Censored (never reacted) rates per condition:')
print(censored.groupby('Condition', observed=True)[['MeleeCensored', 'RangedCensored']].mean().round(3))

## 4 â€” Scaling curves

In [ ]:
SCALING_METRICS = ['AvgFPS', 'AvgCpuMsPerAgent', 'ReactionMeleeMs', 'GoalEntropy']
agg = t.groupby(['Condition', 'AgentCount'], observed=True)[SCALING_METRICS]\
       .agg(['mean', 'std']).reset_index()

fig, axes = plt.subplots(1, len(SCALING_METRICS), figsize=(4 * len(SCALING_METRICS), 3.5))
for ax, m in zip(axes, SCALING_METRICS):
    for cond in CONDITION_ORDER:
        sub = agg[agg['Condition'] == cond].sort_values('AgentCount')
        if sub.empty: continue
        x = sub['AgentCount']
        y = sub[(m, 'mean')]
        s = sub[(m, 'std')].fillna(0)
        ax.plot(x, y, marker='o', color=PALETTE[cond], label=cond)
        ax.fill_between(x, y - s, y + s, alpha=0.2, color=PALETTE[cond])
    ax.set_title(m); ax.set_xlabel('Agent count'); ax.set_xscale('log', base=2)
    ax.set_xticks(sorted(t['AgentCount'].unique()))
    ax.set_xticklabels(sorted(t['AgentCount'].unique()))
axes[0].legend(loc='best', fontsize=8)
fig.suptitle('Scaling curves across architectures (shaded = 1Ïƒ)')
fig.tight_layout()
fig.savefig(FIG_DIR / f'4_scaling_curves_{TS}.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 â€” Trial outcomes

In [ ]:
outcomes = (trials.groupby(['Condition', 'AgentCount', 'Outcome'], observed=True)
                  .size().unstack(fill_value=0))
fig, ax = plt.subplots(figsize=(max(7, 2 * len(sizes)), 4))
outcomes.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
ax.set_ylabel('Trial count')
ax.set_title('Trial outcomes per condition Ã— size')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(FIG_DIR / f'5_outcomes_{TS}.png', dpi=150, bbox_inches='tight')
plt.show()
outcomes

## 6 â€” FPS-over-time sanity check (one representative run per condition Ã— size)

In [ ]:
fig, axes = plt.subplots(len(sizes), 1, figsize=(9, 2.5 * len(sizes)), sharex=True, squeeze=False)
for i, sz in enumerate(sizes):
    ax = axes[i, 0]
    for cond in CONDITION_ORDER:
        sub = perf[(perf['Condition'] == cond) & (perf['AgentCount'] == sz)]
        if sub.empty: continue
        run0 = sub[sub['RunId'] == sub['RunId'].min()]
        ax.plot(run0['Time'], run0['AvgFPS'], color=PALETTE[cond], label=cond, alpha=0.85)
    ax.set_ylabel(f'N={sz}\nAvgFPS')
    ax.axhline(30, color='grey', linestyle='--', linewidth=0.7)
axes[0, 0].legend(loc='best', fontsize=8)
axes[-1, 0].set_xlabel('Trial time (s)')
fig.suptitle('Rolling-average FPS over time â€” run 0 per condition Ã— size')
fig.tight_layout()
fig.savefig(FIG_DIR / f'6_fps_traces_{TS}.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary table (thesis-ready)

In [ ]:
summary = (t.groupby(['Condition', 'AgentCount'], observed=True)[METRICS]
            .agg(['mean', 'std']).round(2))
summary.to_csv(FIG_DIR / f'summary_{TS}.csv')
summary